## 中国主要城市公共预算支出和收入数据分析
### 25PA第四组 第一次小组作业 
- 小组成员：李致豪 刘梓堃 吴雅茵 李雨涵 廖颖鸾 黄瀚霄

1. 数据说明：本次分析的数据来源于中国国家统计局，包含了中国主要城市近20年的 *地方公共预算收入和支出* 、 *地区生产总值* 以及 *地方房地产开发投资额* 和 *地方商品房销售面积* 的数据 。

### 任务2:数据清洗

#### 处理目标

- 统一格式：将原本分散在多个 CSV 文件中的“宽表”数据提取并整合。

- 多维合并：建立以 city（城市）和 year（年度）为联合主键的单一数据集，方便后续计算“财政自给率”（收入/支出）或“财政集中度”（收入/GDP）等衍生指标。

- 结构优化：将数据重塑为“长表”格式，这是 Seaborn、Plotly 等可视化库以及时间序列回归分析的标准输入格式。

In [1]:
import pandas as pd
import os

# 配置路径
raw_data_path = "city_budget_analysis/data_raw/"
clean_data_path = "city_budget_analysis/data_clean/"

def process_city_data(file_name, value_name):
    """
    处理原始CSV文件：跳过前三行，转为长表，清洗年份和数值
    """
    file_path = os.path.join(raw_data_path, file_name)
    
    # 1. 读取数据，跳过前三行元数据
    df = pd.read_csv(file_path, skiprows=3)
    
    # 2. 宽表转长表 (Melt)
    # 保持“地区”列不动，将年份列压缩为“year”列，数值放入 value_name
    df_long = df.melt(id_vars=['地区'], var_name='year', value_name=value_name)
    
    # 3. 清洗年份：将 "2024年" 转为数字 2024
    df_long['year'] = df_long['year'].str.extract('(\10,20\d{2})').astype(float)
    
    # 4. 重命名地区列
    df_long = df_long.rename(columns={'地区': 'city'})
    
    return df_long

# --- 执行清洗与合并 ---

# 1. 分别处理三个核心指标
df_income = process_city_data('city_income.csv', 'income')
df_expend = process_city_data('city_expenditure.csv', 'expend')
df_gdp = process_city_data('gdp.csv', 'gdp')

# 2. 合并数据框 (基于 city 和 year)
# 采用 outer join 确保数据尽可能完整
df_merged = pd.merge(df_income, df_expend, on=['city', 'year'], how='outer')
df_merged = pd.merge(df_merged, df_gdp, on=['city', 'year'], how='outer')

# 3. 处理空值
# 移除 2025 年及其他全为空值的行（根据您的描述，2025年数据多为缺失）
df_merged = df_merged.dropna(subset=['income', 'expend', 'gdp'], how='all')

# 4. 排序：按城市和年份升序排列，方便观察趋势
df_merged = df_merged.sort_values(by=['city', 'year']).reset_index(drop=True)

# 5. 保存清洗后的结果
output_file = os.path.join(clean_data_path, "cleaned_city_budget_data.csv")
df_merged.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✅ 数据清洗完成！已保存至: {output_file}")
print(df_merged.head())

<>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\14036\AppData\Local\Temp\ipykernel_23968\1786589694.py:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df_long['year'] = df_long['year'].str.extract('(\10,20\d{2})').astype(float)
C:\Users\14036\AppData\Local\Temp\ipykernel_23968\1786589694.py:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df_long['year'] = df_long['year'].str.extract('(\10,20\d{2})').astype(float)


FileNotFoundError: [Errno 2] No such file or directory: 'city_budget_analysis/data_raw/city_income.csv'